S7-CT vs S8-TU ECD comparison: Mann-Whitney U, Cliff's delta,
Hodges-Lehmann shift, bootstrap CIs, Holm-Bonferroni correction.

#### OUTPUT
------
  - results_table.csv   : full numeric results, one row per test
  - console summary

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu, norm
from pathlib import Path

In [ ]:
# Load data
data_dir = Path("..") / "Data files" / "Inclusion lists per stage"

df_container = {}
for f in data_dir.glob("*.xlsx"):
    if "S7-CT" in f.name or "S8-TU" in f.name:
        current_df = pd.read_excel(f, sheet_name="NMI data", index_col=0)
        current_df["stage"] = f.name[:5]
        current_df["heat"] = f.stem[-1]
        df_container[f.stem] = current_df

In [ ]:
# ----------------------------------------------------------------------------
# CONFIGURATION
# ----------------------------------------------------------------------------
# Prepare input dataframe
df = pd.concat(df_container.values(), ignore_index=True)
COL_HEAT, COL_STAGE, COL_TYPE, COL_ECD = "heat", "stage", "Type", "ECD (µm)"
df = df[[COL_HEAT, COL_STAGE, COL_TYPE, COL_ECD]]

STAGE_A, STAGE_B = "S7-CT", "S8-TU"      # B is compared against A (B - A)
MIN_ECD = 0.5                             # detection basis, micrometres

# Which groups to test. 'ALL' pools every NMI type within the heat.
TYPE_GROUPS = ["ALL", "Mg-poor Ca-aluminate", "Mg-bearing Ca-aluminate"]

HEATS = ["1", "2", "3", "4"]

ALPHA = 0.05
HL_MAX_PAIRS = 8_000_000                  # thin pairwise diffs above this
FLAG_N = 50                               # flag cells with fewer particles


# ----------------------------------------------------------------------------
# STATISTICS
# ----------------------------------------------------------------------------
def _dominance(x, y):
    """Per-observation dominance counts, computed by sorted-array search
    rather than an n_x * n_y comparison matrix.

    d_i  = mean_j sign(x_i - y_j)   for each x
    d_j  = mean_i sign(x_i - y_j)   for each y
    ties = number of exactly equal pairs (ECD values are rounded, so ties occur)
    """
    xs, ys = np.sort(x), np.sort(y)
    nx, ny = len(xs), len(ys)
    lo = np.searchsorted(ys, xs, "left")
    hi = np.searchsorted(ys, xs, "right")
    d_i = (lo - (ny - hi)) / ny
    ties = int((hi - lo).sum())
    lx = np.searchsorted(xs, ys, "left")
    rx = np.searchsorted(xs, ys, "right")
    d_j = ((nx - rx) - lx) / nx
    return d_i, d_j, ties


def cliffs_delta(x, y, alpha=ALPHA):
    """Cliff's delta with a consistent-variance confidence interval.

    delta = P(x > y) - P(x < y)
      +1 : every x exceeds every y
       0 : complete overlap
      -1 : every x below every y

    The variance estimator follows Cliff (1993) and accounts for the fact that
    the n_x * n_y pairwise comparisons are not independent. No resampling, so
    the interval is reproducible without a random seed.
    """
    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2:
        return np.nan, np.nan, np.nan
    d_i, d_j, ties = _dominance(x, y)
    d = float(d_i.mean())

    # sum over all pairs of (sign_ij - d)^2, from non-tied pair count
    s_pairs = (nx * ny - ties) - nx * ny * d ** 2
    num = (ny ** 2 * np.sum((d_i - d) ** 2)
           + nx ** 2 * np.sum((d_j - d) ** 2)
           - s_pairs)
    den = nx * ny * (nx - 1) * (ny - 1)
    var = num / den if den > 0 else np.nan

    if not np.isfinite(var) or var <= 0:
        return d, np.nan, np.nan
    z = norm.ppf(1 - alpha / 2)
    se = np.sqrt(var)
    lo = d - z * se
    hi = d + z * se
    return d, float(np.clip(lo, -1, 1)), float(np.clip(hi, -1, 1))


def interpret_delta(d):
    """Romano et al. (2006) magnitude conventions."""
    if np.isnan(d):
        return "n/a"
    a = abs(d)
    if a < 0.147:
        return "negligible"
    if a < 0.330:
        return "small"
    if a < 0.474:
        return "medium"
    return "large"


def hodges_lehmann(x, y, alpha=ALPHA):
    """Median of all pairwise differences x_i - y_j, with the distribution-free
    Moses confidence interval.

    This is the location shift that Mann-Whitney actually tests, which is why
    it is preferable to (median(x) - median(y)) as the reported effect in um.
    The interval is read off the order statistics of the pairwise differences
    at a rank set by the normal approximation to U.
    """
    nx, ny = len(x), len(y)
    if nx == 0 or ny == 0:
        return np.nan, np.nan, np.nan
    n_pairs = nx * ny
    if n_pairs > HL_MAX_PAIRS:
        # deterministic thinning: keep every k-th pair via a strided grid
        k = int(np.ceil(n_pairs / HL_MAX_PAIRS))
        diffs = np.subtract.outer(x, y[::k]).ravel()
        n_eff_x, n_eff_y = nx, len(y[::k])
    else:
        diffs = np.subtract.outer(x, y).ravel()
        n_eff_x, n_eff_y = nx, ny

    diffs.sort()
    est = float(np.median(diffs))
    if n_eff_x < 3 or n_eff_y < 3:
        return est, np.nan, np.nan

    m = n_eff_x * n_eff_y
    z = norm.ppf(1 - alpha / 2)
    k_off = m / 2 - z * np.sqrt(n_eff_x * n_eff_y * (n_eff_x + n_eff_y + 1) / 12.0)
    k_idx = int(np.floor(k_off))
    if k_idx < 0:
        return est, float(diffs[0]), float(diffs[-1])
    lo = float(diffs[max(0, k_idx)])
    hi = float(diffs[min(m - 1, m - 1 - k_idx)])
    return est, lo, hi


def holm_bonferroni(pvals, alpha=ALPHA):
    """Step-down correction. Sort p ascending; compare the i-th smallest
    against alpha/(m-i). Once a test fails, it and all larger p-values are
    non-significant. Returns (adjusted_p, reject_flags) in input order."""
    p = np.asarray(pvals, dtype=float)
    m = len(p)
    order = np.argsort(p)
    adj = np.empty(m)
    running = 0.0
    for i, idx in enumerate(order):
        running = max(running, (m - i) * p[idx])   # enforce monotonicity
        adj[idx] = min(running, 1.0)
    return adj, adj < alpha


def run_tests(df):
    rows = []
    for heat in HEATS:
        for grp in TYPE_GROUPS:
            sub = df[df[COL_HEAT] == heat]
            if grp != "ALL":
                sub = sub[sub[COL_TYPE] == grp]

            x = sub.loc[sub[COL_STAGE] == STAGE_B, COL_ECD].to_numpy(float)  # S8
            y = sub.loc[sub[COL_STAGE] == STAGE_A, COL_ECD].to_numpy(float)  # S7

            rec = {
                "heat": heat, "group": grp,
                f"n_{STAGE_A}": len(y), f"n_{STAGE_B}": len(x),
                f"median_{STAGE_A}": np.median(y) if len(y) else np.nan,
                f"median_{STAGE_B}": np.median(x) if len(x) else np.nan,
                f"q3_{STAGE_A}": np.percentile(y, 75) if len(y) else np.nan,
                f"q3_{STAGE_B}": np.percentile(x, 75) if len(x) else np.nan,
            }

            if len(x) < 3 or len(y) < 3:
                rec.update({"U": np.nan, "p_raw": np.nan, "delta": np.nan,
                            "delta_lo": np.nan, "delta_hi": np.nan,
                            "hl_shift": np.nan, "hl_lo": np.nan,
                            "hl_hi": np.nan, "flag": "insufficient n"})
                rows.append(rec)
                continue

            res = mannwhitneyu(x, y, alternative="two-sided")
            d, d_lo, d_hi = cliffs_delta(x, y)
            hl, h_lo, h_hi = hodges_lehmann(x, y)

            rec.update({
                "U": float(res.statistic), "p_raw": float(res.pvalue),
                "delta": d, "delta_lo": d_lo, "delta_hi": d_hi,
                "magnitude": interpret_delta(d),
                "hl_shift": hl, "hl_lo": h_lo, "hl_hi": h_hi,
                "flag": "low n" if min(len(x), len(y)) < FLAG_N else "",
            })
            rows.append(rec)

    out = pd.DataFrame(rows)
    valid = out["p_raw"].notna()
    out["p_adj"] = np.nan
    out["significant"] = False
    if valid.any():
        adj, rej = holm_bonferroni(out.loc[valid, "p_raw"].to_numpy())
        out.loc[valid, "p_adj"] = adj
        out.loc[valid, "significant"] = rej
    return out


def fmt_p(p):
    if np.isnan(p):
        return "n/a"
    return "< 0.001" if p < 0.001 else f"{p:.3f}"

In [ ]:
res = run_tests(df)
print(res)
# res.to_csv("results_table.csv", index=False)